# PyGAD for the Vehicle Routing Problem (VRP)

This notebook demonstrates a purely classical Genetic Algorithm approach to solving the Capacitated Vehicle Routing Problem (CVRP) using `PyGAD`.
It includes the solver, decoding, plotting, and a trial comparison.


## Setup

Install the required packages if needed.


In [ ]:
!pip install pygad matplotlib numpy


In [ ]:
import json
import math
import random
import time
from pathlib import Path
from typing import Any, Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pygad

def euclidean_distance(a: Tuple[float, float], b: Tuple[float, float]) -> float:
    return math.hypot(a[0] - b[0], a[1] - b[1])

def build_distance_matrix(customers: List[Dict[str, Any]]) -> np.ndarray:
    coords = [(float(c["x"]), float(c["y"])) for c in customers]
    n = len(coords)
    dist = np.zeros((n, n), dtype=float)
    for i in range(n):
        for j in range(n):
            dist[i, j] = euclidean_distance(coords[i], coords[j])
    return dist

def route_cost(route: List[int], dist: np.ndarray) -> float:
    if len(route) < 2:
        return 0.0
    return float(sum(dist[route[i], route[i + 1]] for i in range(len(route) - 1)))

def total_cost(routes: List[List[int]], dist: np.ndarray) -> float:
    return float(sum(route_cost(route, dist) for route in routes))

def decode_permutation(
    permutation: List[int],
    demands: Dict[int, int],
    capacity: int,
    max_vehicles: int,
) -> Tuple[List[List[int]], bool]:
    routes: List[List[int]] = []
    current_route: List[int] = [0]
    current_load = 0

    for customer_id in permutation:
        demand = demands[customer_id]
        if demand > capacity:
            return [], False
        if current_load + demand <= capacity:
            current_route.append(customer_id)
            current_load += demand
        else:
            current_route.append(0)
            routes.append(current_route)
            current_route = [0, customer_id]
            current_load = demand
    current_route.append(0)
    routes.append(current_route)
    feasible = len(routes) <= max_vehicles
    return routes, feasible

def ordered_crossover(parent1: np.ndarray, parent2: np.ndarray) -> np.ndarray:
    size = len(parent1)
    child = np.full(size, -1, dtype=int)
    left, right = sorted(random.sample(range(size), 2))
    child[left:right + 1] = parent1[left:right + 1]
    remaining = [gene for gene in parent2 if gene not in child]
    fill_positions = [i for i in range(size) if child[i] == -1]
    for pos, gene in zip(fill_positions, remaining):
        child[pos] = gene
    return child

def crossover_func(parents: np.ndarray, offspring_size: Tuple[int, int], ga_instance) -> np.ndarray:
    offspring = []
    for k in range(offspring_size[0]):
        p1 = parents[k % parents.shape[0]].astype(int)
        p2 = parents[(k + 1) % parents.shape[0]].astype(int)
        offspring.append(ordered_crossover(p1, p2))
    return np.array(offspring, dtype=int)

def mutation_func(offspring: np.ndarray, ga_instance) -> np.ndarray:
    mutated = offspring.copy()
    for i in range(mutated.shape[0]):
        if random.random() < ga_instance.mutation_probability:
            a, b = random.sample(range(mutated.shape[1]), 2)
            mutated[i, a], mutated[i, b] = mutated[i, b], mutated[i, a]
    return mutated

class OfficialPyGADVRP:
    def __init__(
        self,
        instance: Dict[str, Any],
        num_generations: int = 200,
        sol_per_pop: int = 60,
        num_parents_mating: int = 20,
        mutation_probability: float = 0.2,
        seed: int = 42,
    ):
        self.instance_id = instance["instance_id"]
        self.Nv = int(instance["Nv"])
        self.C = int(instance["C"])
        self.raw_customers = instance["customers"]
        self.num_generations = int(num_generations)
        self.sol_per_pop = int(sol_per_pop)
        self.num_parents_mating = int(num_parents_mating)
        self.mutation_probability = float(mutation_probability)
        self.seed = int(seed)
        random.seed(self.seed)
        np.random.seed(self.seed)
        self.customers = self.raw_customers
        self.depot = None
        self.customer_ids: List[int] = []
        self.demands: Dict[int, int] = {}
        for c in self.customers:
            cid = int(c["customer_id"])
            self.demands[cid] = int(c["demand"])
            if cid == 0:
                self.depot = (float(c["x"]), float(c["y"]))
            else:
                self.customer_ids.append(cid)
        if self.depot is None:
            raise ValueError("Depot (customer_id=0) not found in the instance.")
        self.customer_ids.sort()
        self.dist = build_distance_matrix(self.customers)
        self.best_distance = float("inf")
        self.best_routes: List[List[int]] = []
        self.best_valid = False
        self.last_runtime = 0.0

    def initial_population(self) -> np.ndarray:
        base = np.array(self.customer_ids, dtype=int)
        population = []
        for _ in range(self.sol_per_pop):
            candidate = base.copy()
            np.random.shuffle(candidate)
            population.append(candidate)
        return np.array(population, dtype=int)

    def evaluate(self, solution: np.ndarray) -> Tuple[float, float, bool, List[List[int]]]:
        perm = [int(x) for x in solution.tolist()]
        if sorted(perm) != sorted(self.customer_ids):
            penalty_cost = 1e9
            return 1.0 / (1.0 + penalty_cost), penalty_cost, False, []
        routes, feasible = decode_permutation(perm, self.demands, self.C, self.Nv)
        distance = total_cost(routes, self.dist) if routes else 1e9
        if not feasible:
            overflow = max(0, len(routes) - self.Nv)
            penalty_cost = distance + 100000.0 * overflow + 100000.0
            return 1.0 / (1.0 + penalty_cost), penalty_cost, False, routes
        fitness = 1.0 / (1.0 + distance)
        return fitness, distance, True, routes

    def fitness_func(self, ga_instance, solution, solution_idx):
        fitness, distance, valid, routes = self.evaluate(solution)
        if valid and distance < self.best_distance:
            self.best_distance = distance
            self.best_routes = [route[:] for route in routes]
            self.best_valid = True
        return fitness

    def solve(self) -> Dict[str, Any]:
        start = time.perf_counter()
        ga = pygad.GA(
            num_generations=self.num_generations,
            num_parents_mating=self.num_parents_mating,
            fitness_func=self.fitness_func,
            sol_per_pop=self.sol_per_pop,
            num_genes=len(self.customer_ids),
            initial_population=self.initial_population(),
            gene_type=int,
            parent_selection_type="sss",
            keep_parents=4,
            crossover_type=crossover_func,
            mutation_type=mutation_func,
            mutation_probability=self.mutation_probability,
            suppress_warnings=True,
            random_seed=self.seed,
        )
        ga.run()
        best_solution, best_fitness, _ = ga.best_solution()
        _, distance, valid, routes = self.evaluate(best_solution)
        runtime = time.perf_counter() - start
        self.last_runtime = runtime
        return {
            "instance_id": self.instance_id,
            "distance": distance,
            "runtime": runtime,
            "valid": valid,
            "routes": routes,
            "fitness": best_fitness,
        }


In [ ]:
def plot_routes(routes: List[List[int]], customers: List[Dict[str, Any]], title: str = None) -> None:
    coords = {int(c["customer_id"]): (float(c["x"]), float(c["y"])) for c in customers}
    colors = plt.cm.get_cmap("tab10")
    fig, ax = plt.subplots(figsize=(10, 7))
    depot_x, depot_y = coords[0]
    ax.scatter(depot_x, depot_y, marker="X", color="black", s=130, label="Depot")
    for route_index, route in enumerate(routes):
        xs = [coords[node][0] for node in route]
        ys = [coords[node][1] for node in route]
        color = colors(route_index % 10)
        ax.plot(xs, ys, marker="o", color=color, label=f"Route {route_index + 1}")
        for node, x, y in zip(route, xs, ys):
            ax.text(x, y, str(node), fontsize=9, ha="right", va="bottom")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.grid(True, linestyle="--", alpha=0.35)
    if title:
        ax.set_title(title)
    ax.legend(loc="best", fontsize="small", ncol=2)
    plt.show()

def run_trials(
    instance: Dict[str, Any],
    trials: int = 5,
    num_generations: int = 120,
    sol_per_pop: int = 40,
    num_parents_mating: int = 12,
    mutation_probability: float = 0.2,
    base_seed: int = 100,
) -> Tuple[List[float], Dict[str, Any]]:
    distances: List[float] = []
    best_result = None
    for trial_idx in range(1, trials + 1):
        seed = base_seed + trial_idx - 1
        solver = OfficialPyGADVRP(
            instance=instance,
            num_generations=num_generations,
            sol_per_pop=sol_per_pop,
            num_parents_mating=num_parents_mating,
            mutation_probability=mutation_probability,
            seed=seed,
        )
        result = solver.solve()
        distances.append(float(result["distance"]))
        if best_result is None or result["distance"] < best_result["distance"]:
            best_result = result
        print(f"Trial {trial_idx}/{trials}: distance={result["distance"]:.2f}, valid={result["valid"]}, runtime={result["runtime"]:.2f}s")
    return distances, best_result

def plot_distance_summary(distances: List[float], title: str = "Trial distances") -> None:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(range(1, len(distances) + 1), distances, color="tab:blue")
    ax.set_xlabel("Trial")
    ax.set_ylabel("Distance")
    ax.set_title(title)
    ax.set_xticks(range(1, len(distances) + 1))
    ax.grid(True, linestyle="--", alpha=0.4)
    for idx, value in enumerate(distances, start=1):
        ax.text(idx, value + max(distances)*0.01, f"{value:.1f}", ha="center", va="bottom", fontsize=9)
    plt.show()


## Load the final instance file and solve it

This example loads one CVRP instance from `final_instances.json` in the same folder, solves it with PyGAD, and plots the best route.


In [ ]:
instance_path = Path("final_instances.json")
with open(instance_path, "r", encoding="utf-8") as f:
    instances = json.load(f)
instance = instances[0]
print("Loaded instance:", instance["instance_id"], "with", len(instance["customers"]), "nodes")
solver = OfficialPyGADVRP(
    instance=instance,
    num_generations=100,
    sol_per_pop=40,
    num_parents_mating=12,
    mutation_probability=0.2,
    seed=42,
)
result = solver.solve()
print()
print("Best distance:", round(result["distance"], 2))
print("Valid:", result["valid"])
print("Runtime:", f"{result["runtime"]:.2f}s")
print("Route count:", len(result["routes"]))
for idx, route in enumerate(result["routes"], start=1):
    print(f"Route {idx}: {route}")
plot_routes(result["routes"], instance["customers"], title=f"Best solution for {instance["instance_id"]}: {result["distance"]:.1f}")


## Run multiple trials

Run several GA trials to compare solution distance stability.


In [ ]:
distances, best_result = run_trials(
    instance=instance,
    trials=5,
    num_generations=100,
    sol_per_pop=40,
    num_parents_mating=12,
    mutation_probability=0.2,
    base_seed=200,
)
print()
print("Distances:", [round(x, 2) for x in distances])
print("Best trial distance:", round(best_result["distance"], 2))
plot_distance_summary(distances, title=f"PyGAD trial distances for {instance["instance_id"]}")
plot_routes(best_result["routes"], instance["customers"], title=f"Best route from best trial: {best_result["distance"]:.1f}")


## Assistant edit confirmation

This notebook has been updated programmatically by the assistant.


In [ ]:
print("Assistant update: notebook edit confirmed.")
